# 07 — FINAL test (chỉ chạy khi đủ cổng)

**Mở test đúng MỘT lần; chỉ khi N4 đổi `FINAL=True` và Restart & Run All.**
Mọi bảng ngoài mục 7 của protocol phải ghi nhãn **thăm dò (post-hoc)**.

Xem `reports/final_protocol.md` mục 8.
Notebook này dừng ở cổng GATE / PRE-FLIGHT nếu thiếu điều kiện; chưa đọc `test.csv`.

In [1]:
FINAL = False                      # N4 doi thanh True khi chay chinh thuc — ban KHONG doi
PROTOCOL_TAG  = "protocol-v1"
PROTOCOL_MD   = "reports/final_protocol.md"
RESULTS       = "reports/results"
FROZEN        = f"{RESULTS}/frozen_thresholds.csv"
MODELS        = ["dummy", "logreg", "tree", "lightgbm_smotenc", "Tung_mlp_v2"]
LOCKED_HASHES = {  # tu protocol muc 2 — tinh sha256 de doi chieu, KHONG doc noi dung test
    "data/processed/train.csv": "4e75d796b7e993bb350d202d428f9e3ad9d746daa3e42704deb0cbb21482329e",
    "data/processed/val.csv":   "1fcbee5f639ac363bbbf155235d2cad45436d5e3dd7a73b1c1e422289c3485c1",
    "data/processed/test.csv":  "2cac5114304eb69a10b47f34881ded19dfbd5bbcb3d0a9ed107b438c100ae200",
    "models/Tung_mlp_v2.joblib":"5090cbc6273c034ad4446a69b9f8e728217a6a956084de504ca248553759b970",
}

In [2]:
import subprocess
r = subprocess.run(["git", "rev-parse", "--verify", f"refs/tags/{PROTOCOL_TAG}"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("DỪNG: chưa có tag protocol-v1")
    raise SystemExit(1)
PROTOCOL_HASH = r.stdout.strip()
print(f"GATE 1 OK — tag protocol-v1 -> {PROTOCOL_HASH}")

DỪNG: chưa có tag protocol-v1


SystemExit: 1

/home/user/venv-smartstay/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
from pathlib import Path
import pandas as pd

if not Path(PROTOCOL_MD).exists():
    print("DỪNG: thiếu reports/final_protocol.md")
    raise SystemExit(1)
if not Path(FROZEN).exists():
    print("DỪNG: thiếu reports/results/frozen_thresholds.csv")
    raise SystemExit(1)

df = pd.read_csv(FROZEN)
required_models = [m for m in MODELS if m != "dummy"]
RATIOS = {0.5, 1, 2, 3, 5, 10}
missing = []
for m in required_models:
    sub = df[df["model"] == m]
    have = set(sub["r"].astype(float))
    lacking = sorted(RATIOS - have)
    if lacking:
        missing.append((m, lacking))

if missing:
    print("DỪNG: frozen_thresholds.csv thiếu dòng:")
    for m, t in missing:
        print(f"  - {m}: r còn thiếu {t}")
    raise SystemExit(1)
print(f"GATE 2 OK — frozen_thresholds.csv đủ 6 dòng r x {len(required_models)} model")

In [ ]:
import subprocess
r = subprocess.run(["git", "diff", "--quiet", "protocol-v1", "--", "src/", "reports/final_protocol.md"])
if r.returncode != 0:
    r2 = subprocess.run(["git", "diff", "--stat", "protocol-v1", "--", "src/", "reports/final_protocol.md"],
                        capture_output=True, text=True)
    print("DỪNG: src/ hoặc reports/final_protocol.md khác với tag protocol-v1:")
    print(r2.stdout)
    raise SystemExit(1)
print("GATE 3 OK — src/ và reports/final_protocol.md không đổi kể từ protocol-v1")

In [ ]:
import hashlib

for p, expected in LOCKED_HASHES.items():
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    actual = h.hexdigest()
    ok = actual == expected
    print(f"{'OK  ' if ok else 'LECH'} {p}")
    print(f"      ky vong: {expected}")
    print(f"      thuc te: {actual}")
    if not ok:
        print("DỪNG: hash lệch so với LOCKED_HASHES")
        raise SystemExit(1)
print("GATE 4 OK — toàn bộ hash khớp")

In [ ]:
import numpy as np
from pathlib import Path

missing = []
for m in MODELS:
    if m == "dummy":
        continue
    mf, pf = f"models/{m}.joblib", f"models/proba_val_{m}.npy"
    if not Path(mf).exists():
        missing.append(mf)
    if not Path(pf).exists():
        missing.append(pf)
    else:
        try:
            n = len(np.load(pf))
            if n != 14400:
                missing.append(f"{pf} (len={n} != 14400)")
        except Exception as e:
            missing.append(f"{pf} (loi doc: {e})")

if missing:
    print("DỪNG: thiếu model/proba:")
    for x in missing:
        print(f"  - {x}")
    raise SystemExit(1)
print("GATE 5 OK — models/proba đủ (logreg, tree, lightgbm_smotenc, Tung_mlp_v2)")

In [ ]:
from pathlib import Path

if Path("reports/results/test_final.csv").exists():
    print("DỪNG: đã có test_final.csv — không chạy lại")
    raise SystemExit(1)
print("GATE 6 OK — chưa có reports/results/test_final.csv")

In [ ]:
if FINAL is False:
    print("PRE-FLIGHT PASSED. Chỉ N4 mới được đổi FINAL=True + Restart & Run All.")
    raise SystemExit